# EDA nhanh — `obt_match_360` & `obt_player_360`

Notebook kiểm tra nhanh 2 bảng OBT (One Big Table) vừa xây từ Silver — **KHÔNG phải** notebook EDA
Feature/ML (đã có ở `football_gold_features_eda.ipynb`). Mục tiêu chỉ là xác nhận:
- Grain đúng (`obt_match_360`: 1 dòng/match_id; `obt_player_360`: 1 dòng/player/season)
- Độ phủ (coverage) của từng nhóm thông tin ghép vào
- Vài dòng mẫu để soát bằng mắt


In [ ]:
# [DÀNH RIÊNG CHO GOOGLE COLAB]
import sys, os
os.environ['MINIO_ENDPOINT'] = 'http://20.41.113.183:9000'
os.environ['MINIO_ACCESS_KEY'] = 'minioadmin'
os.environ['MINIO_SECRET_KEY'] = 'minioadmin123'
if 'SEED_PATH' in os.environ: del os.environ['SEED_PATH']
if 'google.colab' in sys.modules:
    !rm -rf /content/Lab
    !git clone -b ml https://github.com/nbngoc123/Lab.git /content/Lab
    os.chdir('/content/Lab/football-lake')
    sys.path.insert(0, '/content/Lab/football-lake')
    !pip install minio duckdb pandas boto3 python-dotenv -q
    print('Setup xong Colab!')


In [ ]:
import io
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

from lake import minio_io as mio

def read_gold(key):
    return pd.read_parquet(io.BytesIO(mio.read_bytes(key)))

obt_match = read_gold("gold/obt/obt_match_360.parquet")
obt_player = read_gold("gold/obt/obt_player_360.parquet")
print("obt_match_360 :", obt_match.shape)
print("obt_player_360:", obt_player.shape)


## 1. `obt_match_360` — kiểm tra Grain & mẫu

In [ ]:
dup = obt_match["match_id"].duplicated().sum()
print("Số dòng trùng match_id:", dup, "-> Grain OK" if dup == 0 else "-> LỖI GRAIN")
display(obt_match.head(5))


In [ ]:
cols_group = {
    "Kết quả": ["home_goals", "away_goals", "result", "total_goals"],
    "Odds (đã tổng hợp)": ["odds_home_avg", "odds_draw_avg", "odds_away_avg", "n_bookmakers"],
    "Thời tiết": ["temp_c", "precip_mm", "wind_kmh"],
    "xG (chính trận đó)": ["home_xg", "away_xg"],
    "Stats API-Football (chính trận đó)": ["home_possession_pct", "away_possession_pct"],
}
cov = {name: obt_match[cols].notna().mean().mean() for name, cols in cols_group.items()}
display(pd.Series(cov, name="% non-null").to_frame())


## 2. `obt_player_360` — kiểm tra Grain & mẫu

In [ ]:
dup_p = obt_player.duplicated(["player_id_af", "season"]).sum()
print("Số dòng trùng (player_id_af, season):", dup_p, "-> Grain OK" if dup_p == 0 else "-> LỖI GRAIN")
display(obt_player.head(5))


In [ ]:
cols_group_p = {
    "Cơ bản (af_players)": ["appearances", "minutes", "goals", "assists", "rating"],
    "xG/xA (Understat, ghép theo tên)": ["xg", "xa", "npxg"],
    "FPL (giá/độ phổ biến, ghép theo tên)": ["price_m", "selected_by_percent", "fpl_total_points"],
}
cov_p = {name: obt_player[cols].notna().mean().mean() for name, cols in cols_group_p.items()}
display(pd.Series(cov_p, name="% non-null").to_frame())

print("\nLưu ý: cột ghép theo TÊN đã chuẩn hóa (không có player_id chung giữa af/understat/fpl).")
print("Nếu coverage thấp, xem gold/qa/obt_player_360_unmatched.csv để biết tên nào chưa ghép được.")


## 3. Kết luận nhanh

- Nếu cả 2 bảng Grain OK (0 dòng trùng) và coverage các nhóm nguồn chính (kết quả trận, af_players)
  đạt gần 100% thì OBT đã sẵn sàng cho BI/phân tích.
- Coverage thấp ở nhóm ghép theo TÊN (xG Understat, FPL trong `obt_player_360`) là điều cần theo dõi
  thêm — khác với lỗi join, đây là giới hạn chưa có `seed/player_alias.csv` chuẩn như đội bóng.
